In [ ]:

# ============================================================
# USER STORY 2: GIÁM SÁT AN TOÀN VÀ CHẨN ĐOÁN RỦI RO NHIỆT
# Safety & Risk Monitor
# ============================================================

# 1. KHUNG THIẾT KẾ WHAT - WHY - HOW
#
# WHAT - Dữ liệu:
# - Chuỗi nhiệt độ dầu OT với ngưỡng cảnh báo 85°C
#   và ngưỡng nguy hiểm 95°C.
# - Danh sách sự cố, thời lượng quá nhiệt và Degree-Hours.
# - Đường cong LOESS và Z-score biến động phụ tải.
#
# WHY - Tác vụ phân tích:
# - Phát hiện thời điểm nhiệt độ vượt ngưỡng nguy hiểm.
# - Xác định cụm sự cố theo dòng thời gian.
# - Truy tìm nhánh phụ tải có liên quan đến quá nhiệt.
#
# HOW - Phương pháp trực quan hóa:
# - Threshold Line Chart: biểu diễn nhiệt độ và ngưỡng an toàn.
# - Barcode / Rug Anomaly Strip: xác định vị trí sự cố.
# - LOESS Regression: phân tích quan hệ phi tuyến giữa tải và OT.
#
# ACTION TITLES:
# 1. Nhiệt độ OT vi phạm ngưỡng cảnh báo 85°C trong mùa hè.
# 2. Các sự cố nguy hiểm có thể tập trung vào tháng 7 và tháng 8.
# 3. Phân tích phản ứng nhiệt khi phụ tải HUFL tăng cao.


# ============================================================
# 2. KHAI BÁO THƯ VIỆN
# ============================================================

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# 3. THIẾT LẬP ĐƯỜNG DẪN VÀ NẠP DỮ LIỆU
# ============================================================

BASE_DIR = os.path.abspath("..")

ARTIFACTS_DIR = os.path.join(
    BASE_DIR,
    "output",
    "artifacts",
    "ETTh1"
)

# Nạp dữ liệu chuỗi thời gian
clean_df = pd.read_parquet(
    os.path.join(ARTIFACTS_DIR, "clean_ett.parquet")
)

# Nạp danh sách sự cố nhiệt
breaches_df = pd.read_parquet(
    os.path.join(ARTIFACTS_DIR, "thermal_breach_events.parquet")
)

# Nạp dữ liệu đường cong LOESS đã tính toán
loess_df = pd.read_parquet(
    os.path.join(ARTIFACTS_DIR, "loess_curves.parquet")
)

# Chuyển cột ngày sang kiểu datetime nếu cần
clean_df["date"] = pd.to_datetime(clean_df["date"])

print(
    "Tổng số đợt sự cố nhiệt độ dầu >= 85°C:",
    len(breaches_df)
)

print("\nNăm dòng dữ liệu sự cố đầu tiên:")
print(breaches_df.head())


# ============================================================
# 4. KHÁM PHÁ 1: THRESHOLD CHART VÀ BARCODE ANOMALY STRIP
# ============================================================

fig_sec = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.8, 0.2],
    vertical_spacing=0.06,
    subplot_titles=[
        "Nhiệt độ dầu OT và các ngưỡng cảnh báo",
        "Vị trí các sự cố quá nhiệt nguy hiểm theo thời gian"
    ]
)

# 4.1. Biểu diễn nhiệt độ dầu OT thực tế
fig_sec.add_trace(
    go.Scatter(
        x=clean_df["date"],
        y=clean_df["OT"],
        mode="lines",
        line=dict(color="#1E3A8A", width=1.5),
        name="Nhiệt độ dầu OT"
    ),
    row=1,
    col=1
)

# 4.2. Ngưỡng cảnh báo 85°C
fig_sec.add_hline(
    y=85,
    line_dash="dash",
    line_color="#D97706",
    annotation_text="Cảnh báo (85°C)",
    row=1,
    col=1
)

# 4.3. Ngưỡng nguy hiểm 95°C
fig_sec.add_hline(
    y=95,
    line_dash="dot",
    line_color="#DC2626",
    annotation_text="Nguy hiểm (95°C)",
    row=1,
    col=1
)

# 4.4. Xác định các điểm nhiệt độ vượt ngưỡng nguy hiểm
crit_pts = clean_df[
    clean_df["OT"] >= 95.0
].copy()

# Đánh dấu các điểm vượt ngưỡng 95°C
fig_sec.add_trace(
    go.Scatter(
        x=crit_pts["date"],
        y=crit_pts["OT"],
        mode="markers",
        marker=dict(
            color="#DC2626",
            size=6
        ),
        name="Vi phạm >= 95°C"
    ),
    row=1,
    col=1
)

# 4.5. Barcode Anomaly Strip
fig_sec.add_trace(
    go.Scatter(
        x=crit_pts["date"],
        y=np.ones(len(crit_pts)),
        mode="markers",
        marker=dict(
            symbol="line-ns",
            color="#DC2626",
            size=16,
            line=dict(width=2)
        ),
        name="Vị trí sự cố (Rug Strip)"
    ),
    row=2,
    col=1
)

# 4.6. Thiết lập giao diện biểu đồ
fig_sec.update_layout(
    title="Giám sát nhiệt độ dầu và xác định sự cố quá nhiệt",
    template="plotly_white",
    height=600,
    hovermode="x unified"
)

fig_sec.update_yaxes(
    title_text="Nhiệt độ (°C)",
    row=1,
    col=1
)

fig_sec.update_yaxes(
    showticklabels=False,
    title_text="Sự cố",
    row=2,
    col=1
)

fig_sec.update_xaxes(
    title_text="Thời gian",
    row=2,
    col=1
)

fig_sec.show()


# ============================================================
# 5. KHÁM PHÁ 2: ĐƯỜNG CONG HỒI QUY PHI TUYẾN LOESS
# ============================================================

# Lọc dữ liệu mùa hè, biến HUFL và chế độ nhiệt All
sub_loess = loess_df[
    (loess_df["season"] == "Summer")
    & (loess_df["feature"] == "HUFL")
    & (loess_df["thermal_regime"] == "All")
].copy()

# Sắp xếp theo giá trị phụ tải để đường cong hiển thị đúng
sub_loess = sub_loess.sort_values("feature_val")

# Vẽ đường cong LOESS đã được tính toán trước
fig_loess = px.line(
    sub_loess,
    x="feature_val",
    y="loess_ot",
    color_discrete_sequence=["#DC2626"],
    labels={
        "feature_val": "Công suất tải Cao áp Hữu ích HUFL (MW)",
        "loess_ot": "Nhiệt độ dầu làm mịn LOESS (°C)"
    },
    title=(
        "Phân tích quan hệ phi tuyến giữa phụ tải HUFL "
        "và nhiệt độ dầu OT trong mùa hè"
    )
)

fig_loess.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig_loess.show()
